# 🎮 Face Recognition Game: "Who are you?"
**กติกาของเกม:**
1. เกมจะสุ่ม "ชื่อเป้าหมาย" มาให้คุณ (เช่น Target: Captun)
2. คุณต้องเอาหน้าตัวเอง (หรือเอารูปเพื่อนในมือถือมาจ่อกล้อง) ให้ AI ทายให้ตรงกับชื่อเป้าหมาย
3. ต้องทำให้ความมั่นใจ (Confidence) ถึง **90%** ขึ้นไป ถึงจะผ่านด่าน!
4. มีเวลาจำกัดด่านละ 15 วินาที ทำให้ได้คะแนนเยอะที่สุด!

⚠️ **คำเตือน:** เกมนี้ต้องใช้ความลื่นไหลของกล้อง (FPS สูง) แนะนำให้รันไฟล์นี้บน **เครื่องคอมพิวเตอร์ของคุณเอง (Local)** แทน Google Colab ครับ


In [ ]:
# 1. ติดตั้งไลบรารีที่จำเป็น (ถ้ารันบนเครื่องตัวเอง)
# !pip install facenet-pytorch scikit-learn joblib opencv-python Pillow numpy


In [ ]:
# 2. โหลดโมเดล AI
import cv2
import torch
import numpy as np
import joblib
import time
import random
import os
from facenet_pytorch import MTCNN, InceptionResnetV1
from PIL import Image

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'💻 ใช้หน่วยประมวลผล: {device}')

print('กำลังโหลดโมเดล FaceNet...')
mtcnn = MTCNN(image_size=160, margin=20, keep_all=True, select_largest=False, post_process=True, device=device)
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

# โหลด SVM Model
clf_path = 'facenet_svm_model.pkl'
le_path = 'label_encoder.pkl'

if os.path.exists(clf_path) and os.path.exists(le_path):
    clf = joblib.load(clf_path)
    le = joblib.load(le_path)
    all_names = list(le.classes_)
    print('✅ โหลดโมเดลสำเร็จ! พร้อมลุย!')
else:
    raise FileNotFoundError('❌ ไม่พบไฟล์โมเดล กรุณาก๊อปปี้ facenet_svm_model.pkl และ label_encoder.pkl มาไว้ที่เดียวกับไฟล์นี้')


In [ ]:
# 3. ระบบเกมหลัก
def play_face_game():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("❌ ไม่สามารถเปิดกล้องได้")
        return

    # สถานะเกม
    score = 0
    time_limit = 15.0
    target_name = random.choice(all_names)
    start_time = time.time()
    game_over = False

    print("🎮 เริ่มเกม! กด [Q] เพื่อออกจากเกม")

    while True:
        ret, frame = cap.read()
        if not ret: break
        
        frame = cv2.flip(frame, 1) # กลับซ้ายขวาเหมือนกระจก
        display_frame = frame.copy()
        
        if game_over:
            cv2.putText(display_frame, "GAME OVER!", (50, 200), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 5)
            cv2.putText(display_frame, f"Final Score: {score}", (50, 280), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 255), 3)
            cv2.putText(display_frame, "Press 'Q' to Quit or 'R' to Restart", (50, 350), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            cv2.imshow("Face Game", display_frame)
            
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'): break
            if key == ord('r'):
                score = 0
                target_name = random.choice(all_names)
                start_time = time.time()
                game_over = False
            continue

        elapsed = time.time() - start_time
        time_left = max(0, time_limit - elapsed)
        
        if time_left == 0:
            game_over = True

        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(img_rgb)
        
        boxes, probs = mtcnn.detect(pil_img)
        
        highest_conf_for_target = 0.0

        if boxes is not None:
            faces = mtcnn(pil_img)
            if faces is not None:
                faces_tensor = faces.to(device)
                if faces_tensor.dim() == 3:
                    faces_tensor = faces_tensor.unsqueeze(0)
                    
                with torch.no_grad():
                    embeddings = resnet(faces_tensor).cpu().numpy()
                
                preds = clf.predict(embeddings)
                pred_probs = clf.predict_proba(embeddings)
                
                for i, box in enumerate(boxes):
                    if i >= len(preds) or probs[i] < 0.90: continue
                    w, h = box[2] - box[0], box[3] - box[1]
                    if w < 40 or h < 40: continue
                    
                    prob_max = np.max(pred_probs[i])
                    name = le.inverse_transform([preds[i]])[0]
                    
                    if name == target_name:
                        if prob_max > highest_conf_for_target:
                            highest_conf_for_target = prob_max
                    
                    color = (0, 255, 255) if name == target_name else (200, 200, 200)
                    x1, y1, x2, y2 = map(int, box)
                    cv2.rectangle(display_frame, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(display_frame, f"{name} {prob_max*100:.0f}%", (x1, max(0, y1-10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        # ผ่านด่าน!
        if highest_conf_for_target >= 0.90:
            score += 1
            cv2.rectangle(display_frame, (0,0), (display_frame.shape[1], display_frame.shape[0]), (0,255,0), 15)
            cv2.putText(display_frame, "SUCCESS!", (150, 250), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 5)
            cv2.imshow("Face Game", display_frame)
            cv2.waitKey(500) # โชว์ข้อความแป๊บเดียว
            
            target_name = random.choice(all_names)
            start_time = time.time()

        # วาด UI
        cv2.putText(display_frame, f"TARGET: {target_name}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 3)
        cv2.putText(display_frame, f"Time: {time_left:.1f}s", (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255) if time_left < 5 else (255, 255, 255), 2)
        cv2.putText(display_frame, f"Score: {score}", (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
        
        # Progress bar
        bar_x, bar_y, bar_w, bar_h = 20, 400, 400, 30
        cv2.rectangle(display_frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (50, 50, 50), -1)
        fill_w = int(bar_w * highest_conf_for_target)
        bar_color = (0, 255, 0) if highest_conf_for_target >= 0.90 else (0, 165, 255)
        cv2.rectangle(display_frame, (bar_x, bar_y), (bar_x + fill_w, bar_y + bar_h), bar_color, -1)
        cv2.rectangle(display_frame, (bar_x, bar_y), (bar_x + bar_w, bar_y + bar_h), (255, 255, 255), 2)
        
        line_90_x = bar_x + int(bar_w * 0.90)
        cv2.line(display_frame, (line_90_x, bar_y - 5), (line_90_x, bar_y + bar_h + 5), (0, 0, 255), 2)
        cv2.putText(display_frame, "90%", (line_90_x - 15, bar_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

        cv2.imshow("Face Game", display_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# เรียกใช้งานเกม
play_face_game()
